In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)

plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.family": "Time New Roman",
    "font.size": 12,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

from vs_ns_periodic_mrSAV_solver import vs_mrSAV_Vorticity_Stream_Periodic_Solver as vs_mrSAV_solver

In [ ]:
nu = 1/50
m = 1
gam = 1000

s_domain = (0,0, 2*np.pi, 2*np.pi)
discrete_num = [128,128]
xn = np.linspace(s_domain[0],s_domain[2],discrete_num[0]+1)
yn = np.linspace(s_domain[1],s_domain[3],discrete_num[1]+1)
X,Y = np.meshgrid(xn,yn)

t_period = (0, 1)
delta = 0.01

scheme_configs = [
    ("ETDMS2", "ETD-MS2", '#009E73', 'x', '-.'),
    ("ETD_mrGSAV_MS2_b", "ETD-mr-SAV-MS2o", '#0072B2', 'o', '-'),
    ("mr_SAV_BDF2", "mr-SAV-BDF2", '#D55E00', 's', '--'),
]

def sanitize_error_value(value, blowup_value=None):
    if np.isfinite(value):
        return value
    if blowup_value is not None:
        return blowup_value
    return value

def err_test(Re, gam, exponents, step_method, initial_data, force_term, reference_data,
             perturb_ratio=0.0, rng_seed=0, blowup_value=None):
    ref_solution_omega = reference_data["omega"]
    ref_solution_psi = reference_data["psi"]
    ref_solution_u = reference_data["u"]
    ref_solution_v = reference_data["v"]

    exponents_used = []
    step_counts = []
    taus = []
    err_omega = []
    err_psi = []
    err_u = []
    err_q = []

    for idx, exponent in enumerate(exponents):
        solver = vs_mrSAV_solver(Re, gam, s_domain, discrete_num, initial_data, force_term, step_method)
        N = int(100 * 2**exponent)
        tau_mean = (t_period[1] - t_period[0]) / N

        if perturb_ratio > 0:
            rng = np.random.default_rng(rng_seed + idx)
            perturbation = rng.uniform(-1, 1, N)
            perturbation = perturbation - np.mean(perturbation)
            tau = tau_mean * (1 + perturb_ratio * perturbation)
        else:
            tau = np.full(N, tau_mean, dtype=np.float64)

        solver.solve_given_tau(t_period, tau)
        omega = solver.Omega[-1]
        psi = solver.vorticity2stream(omega)
        u, v = solver.stream2velocity(psi)
        q = solver.q[-1]

        exponents_used.append(exponent)
        step_counts.append(N)
        taus.append(np.mean(tau))
        err_omega_value = np.sqrt(solver.h * np.mean(np.square(omega - ref_solution_omega)))
        err_psi_value = np.sqrt(solver.h * np.mean(np.square(psi - ref_solution_psi)))
        err_u_value = np.sqrt(solver.h * np.mean(np.square(u - ref_solution_u)) + solver.h * np.mean(np.square(v - ref_solution_v)))
        err_q_value = np.abs(q - 1)

        err_omega.append(sanitize_error_value(err_omega_value, blowup_value=blowup_value))
        err_psi.append(sanitize_error_value(err_psi_value, blowup_value=blowup_value))
        err_u.append(sanitize_error_value(err_u_value, blowup_value=blowup_value))
        err_q.append(sanitize_error_value(err_q_value, blowup_value=blowup_value))

    return {
        "exponent": np.array(exponents_used),
        "N": np.array(step_counts),
        "tau": np.array(taus),
        "omega": np.array(err_omega),
        "psi": np.array(err_psi),
        "u": np.array(err_u),
        "q": np.array(err_q),
    }

def run_convergence_experiment(Re, gam, exponents, initial_data, force_term, reference_data,
                               perturb_ratio=0.0, rng_seed=2026, blowup_value=None):
    results = {}
    for offset, (step_method, label, _, _, _) in enumerate(scheme_configs):
        print(label)
        results[label] = err_test(
            Re, gam, exponents, step_method, initial_data, force_term, reference_data,
            perturb_ratio=perturb_ratio, rng_seed=rng_seed + 1000 * offset, blowup_value=blowup_value,
        )
    return results

def print_convergence_rates(result, label, component="omega"):
    taus = result["tau"]
    errs = result[component]
    for i, (tau, err) in enumerate(zip(taus, errs)):
        if i == 0:
            print(f"tau = {tau:.2e}, err = {err:.2e}")
        else:
            if np.isfinite(errs[i - 1]) and np.isfinite(err) and errs[i - 1] > 0 and err > 0:
                rate = np.log(errs[i - 1] / err) / np.log(taus[i - 1] / tau)
                print(f"tau = {tau:.2e}, err = {err:.2e}, rate = {rate:.2f}")
            else:
                print(f"tau = {tau:.2e}, err = {err:.2e}, rate = blow-up")

def plot_convergence_results(results, save_path, title=None, x_key="tau"):
    from matplotlib.ticker import LogLocator

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.6), dpi=300)

    panel_specs = [
        dict(key="u", ylabel=r'$L^2$ error for velocity'),
        dict(key="omega", ylabel=r'$L^2$ error for vorticity'),
        dict(key="q", ylabel=r'$|r|$'),
    ]

    ref_label_for_q = "ETD-mr-SAV-MS2o"
    ref_label_for_second_order = "mr-SAV-BDF2"
    if x_key == "tau":
        x_label = r'time step size ($\tau$)'
    elif x_key == "N":
        x_label = r'number of time steps ($N$)'
    else:
        raise ValueError(f"Unsupported x_key: {x_key}")

    for ax, spec in zip(axes, panel_specs):
        for _, label, color, marker, ls in scheme_configs:
            x_values = results[label][x_key]
            errs = results[label][spec["key"]]
            exponents = results[label]["exponent"]
            integer_mask = np.array([np.isclose(exponent, round(exponent)) for exponent in exponents])
            ax.loglog(x_values, errs, color=color, linestyle=ls, linewidth=1.2, label=label)
            ax.loglog(
                x_values[integer_mask], errs[integer_mask],
                color=color, marker=marker, linestyle='None', markersize=5.2,
                markerfacecolor='white', markeredgewidth=1.5,
            )

        x_ref = results[ref_label_for_second_order][x_key]
        taus_ref = results[ref_label_for_second_order]["tau"]
        errs_ref = results[ref_label_for_second_order][spec["key"]]
        if x_key == "tau":
            c2 = errs_ref[-3] / taus_ref[-3]**2 * 0.2
            ax.loglog(x_ref, c2 * taus_ref**2, color='0.3', linestyle=':', linewidth=1.3, label=r'$\mathcal{O}(\tau^2)$', zorder=0)
        else:
            c2 = errs_ref[-3] * x_ref[-3]**2 * 0.2
            ax.loglog(x_ref, c2 / x_ref**2, color='0.3', linestyle=':', linewidth=1.3, label=r'$\mathcal{O}(N^{-2})$', zorder=0)

        if spec["key"] == "q":
            x_q = results[ref_label_for_q][x_key]
            taus_q = results[ref_label_for_q]["tau"]
            errs_q = results[ref_label_for_q]["q"]
            if x_key == "tau":
                c1 = errs_q[-3] / taus_q[-3] * 0.5
                ax.loglog(x_q, c1 * taus_q, color='0.3', linestyle='--', linewidth=1.3, label=r'$\mathcal{O}(\tau)$', zorder=0)
            else:
                c1 = errs_q[-3] * x_q[-3] * 0.5
                ax.loglog(x_q, c1 / x_q, color='0.3', linestyle='--', linewidth=1.3, label=r'$\mathcal{O}(N^{-1})$', zorder=0)

        ax.set_ylim(1e-10, 1e-0)
        ax.set_xlabel(x_label)
        ax.set_ylabel(spec['ylabel'])
        ax.grid(True, which='major', ls='--', lw=0.45, alpha=0.55, color='gray')
        ax.grid(True, which='minor', ls=':', lw=0.3, alpha=0.35, color='gray')
        ax.tick_params(which='both', direction='in', top=True, right=True, labelsize=10)
        ax.xaxis.set_minor_locator(LogLocator(base=10, subs='auto', numticks=10))
        ax.yaxis.set_minor_locator(LogLocator(base=10, subs='auto', numticks=10))

    handles, labels = axes[2].get_legend_handles_labels()
    seen = set()
    unique_handles, unique_labels = [], []
    for handle, label in zip(handles, labels):
        if label not in seen:
            seen.add(label)
            unique_handles.append(handle)
            unique_labels.append(label)

    fig.legend(
        unique_handles, unique_labels,
        loc='lower center', bbox_to_anchor=(0.5, 0), ncol=len(unique_handles),
        fontsize=9, framealpha=0.95, edgecolor='0.75', fancybox=False,
        handlelength=2.2, columnspacing=1.8, handletextpad=0.5,
    )
    if title is not None:
        fig.suptitle(title, y=1.02, fontsize=12)
        fig.tight_layout(rect=[0, 0.13, 1, 0.97], pad=0.5, w_pad=1.0)
    else:
        fig.tight_layout(rect=[0, 0.13, 1, 1], pad=0.5, w_pad=1.0)
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
def _is_blowup_error(value, blowup_value=1e6):
    return (not np.isfinite(value)) or value <= 0 or np.isclose(value, blowup_value)


def _format_error_for_table(value, blowup_value=1e6):
    if _is_blowup_error(value, blowup_value=blowup_value):
        return r"\mathrm{blow\mbox{-}up}"
    return f"{value:.2e}"


def _format_order_for_table(prev_err, err, prev_tau, tau, blowup_value=1e6):
    if (
        _is_blowup_error(prev_err, blowup_value=blowup_value)
        or _is_blowup_error(err, blowup_value=blowup_value)
        or prev_tau <= 0
        or tau <= 0
        or np.isclose(prev_tau, tau)
    ):
        return "--"
    order = np.log(prev_err / err) / np.log(prev_tau / tau)
    if not np.isfinite(order):
        return "--"
    return f"{order:.2f}"


def make_convergence_latex_table(results, caption, label, blowup_value=1e6):
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\begin{tabular}{llrrrrrrrl}",
        r"\toprule",
        r"Scheme & $k$ & $N$ & $\tau$ & $e_u$ & order & $e_\omega$ & order & $e_r$ & order \\",
        r"\midrule",
    ]

    scheme_labels = [label for _, label, _, _, _ in scheme_configs]
    for scheme_idx, scheme_label in enumerate(scheme_labels):
        result = results[scheme_label]
        exponents = result["exponent"]
        integer_mask = np.array([np.isclose(exponent, round(exponent)) for exponent in exponents])
        row_indices = np.where(integer_mask)[0]

        prev_values = None
        for row_idx, idx in enumerate(row_indices):
            k = int(round(result["exponent"][idx]))
            N = int(result["N"][idx])
            tau = result["tau"][idx]
            err_u = result["u"][idx]
            err_omega = result["omega"][idx]
            err_r = result["q"][idx]

            if prev_values is None:
                order_u = order_omega = order_r = "--"
            else:
                prev_tau, prev_u, prev_omega, prev_r = prev_values
                order_u = _format_order_for_table(prev_u, err_u, prev_tau, tau, blowup_value=blowup_value)
                order_omega = _format_order_for_table(prev_omega, err_omega, prev_tau, tau, blowup_value=blowup_value)
                order_r = _format_order_for_table(prev_r, err_r, prev_tau, tau, blowup_value=blowup_value)

            scheme_cell = scheme_label if row_idx == 0 else ""
            lines.append(
                f"{scheme_cell} & {k:d} & {N:d} & {tau:.2e} & "
                f"{_format_error_for_table(err_u, blowup_value=blowup_value)} & {order_u} & "
                f"{_format_error_for_table(err_omega, blowup_value=blowup_value)} & {order_omega} & "
                f"{_format_error_for_table(err_r, blowup_value=blowup_value)} & {order_r} \\\\" 
            )
            prev_values = (tau, err_u, err_omega, err_r)

        if scheme_idx < len(scheme_labels) - 1:
            lines.append(r"\midrule")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


# ── Visual display table (pandas) ──────────────────────────────────────────
BLOWUP_SENTINEL = 1e15  # replaces NaN/inf to signal numerical blow-up

def make_convergence_display_table(results, exp_range=(1.0, 3.0), blowup_value=1e6, sentinel=BLOWUP_SENTINEL):
    """Return a pandas DataFrame for all exponents in [exp_lo, exp_hi]."""
    import pandas as pd

    exp_lo, exp_hi = exp_range
    rows = []
    scheme_labels = [lbl for _, lbl, _, _, _ in scheme_configs]

    for scheme_label in scheme_labels:
        result = results[scheme_label]
        exponents = np.array(result["exponent"])
        mask = (exponents >= exp_lo - 1e-9) & (exponents <= exp_hi + 1e-9)
        indices = np.where(mask)[0]

        prev_tau = prev_u = prev_omega = prev_r = None
        for idx in indices:
            k_raw     = result["exponent"][idx]
            N         = int(result["N"][idx])
            tau       = result["tau"][idx]
            err_u     = result["u"][idx]
            err_omega = result["omega"][idx]
            err_r     = result["q"][idx]

            def sanitize(v):
                return sentinel if (not np.isfinite(v) or v <= 0 or np.isclose(v, blowup_value)) else v

            err_u     = sanitize(err_u)
            err_omega = sanitize(err_omega)
            err_r     = sanitize(err_r)

            def calc_order(pe, ce, pt, ct):
                if pe is None or pe >= sentinel * 0.9 or ce >= sentinel * 0.9:
                    return float("nan")
                if pt <= 0 or ct <= 0 or np.isclose(pt, ct):
                    return float("nan")
                o = np.log(pe / ce) / np.log(pt / ct)
                return o if np.isfinite(o) else float("nan")

            rows.append({
                "Scheme":    scheme_label,
                "k (exp)":   f"{k_raw:.1f}",
                "N":         N,
                "tau":       tau,
                "e_u":       err_u,
                "ord_u":     calc_order(prev_u,    err_u,     prev_tau, tau),
                "e_omega":   err_omega,
                "ord_omega": calc_order(prev_omega, err_omega, prev_tau, tau),
                "e_r":       err_r,
                "ord_r":     calc_order(prev_r,     err_r,     prev_tau, tau),
            })
            prev_tau, prev_u, prev_omega, prev_r = tau, err_u, err_omega, err_r

    return pd.DataFrame(rows)


In [ ]:
def force_term(X, Y, t):
    f = m*np.cos(m*X)
    return f

def initial_streamfunction(x: np.ndarray, y: np.ndarray, nu: float, m: float, eps: float) -> np.ndarray:
    """
    计算初始流函数 φ(0)
    参数:
        x, y: 网格坐标数组 (可以是任意维度，会自动广播)
        nu: 运动粘度 ν
        m: 基波波数 m
        eps: 扰动强度 ε
    返回:
        phi: 流函数场，形状与 x, y 一致
    """
    # 1. 基流项: -1/(ν m³) * cos(m y)
    # base_flow = - (1.0 / (nu * m**3)) * np.cos(m * y)
    base_flow = np.zeros_like(x)
    
    # 2. 生成所有满足 |k| ≤ 10 的二维整数波矢 (k1, k2)
    k_max = 10
    k1_vals = np.arange(-k_max, k_max + 1)
    k2_vals = np.arange(-k_max, k_max + 1)
    k1_grid, k2_grid = np.meshgrid(k1_vals, k2_vals, indexing="ij")
    
    # 计算波矢模长 |k| = sqrt(k1² + k2²)
    k_mod = np.sqrt(k1_grid**2 + k2_grid**2)
    
    # 过滤 |k| ≤ 10 的波矢（排除模长>10的点）
    mask = k_mod <= 10
    k1_valid = k1_grid[mask]
    k2_valid = k2_grid[mask]
    k_mod_valid = k_mod[mask]
    
    # 3. 计算扰动项求和
    perturbation = np.zeros_like(x, dtype=np.float64)
    for k1, k2, k_abs in zip(k1_valid, k2_valid, k_mod_valid):
        if k_abs < 1e-10: # 避免 |k|=0 时分母为0
            continue
        # 1/|k|^(5/3) * cos(k1 x) * cos(k2 y)
        term = (1 / (k_abs ** (3))) *(    1*np.cos(k1 * x) * np.cos(k2 * y) 
                                        + 1*np.sin(k1 * x) * np.cos(k2 * y) 
                                        + 1*np.cos(k1 * x) * np.sin(k2 * y) 
                                        + 1*np.sin(k1 * x) * np.sin(k2 * y))
        perturbation += term
    
    # 4. 总流函数 = 基流 + ε*扰动
    phi = base_flow + eps * perturbation
    return phi

initial_phi = initial_streamfunction(X[:-1, :-1], Y[:-1, :-1], nu, m, 2.5)
solver_init = vs_mrSAV_solver(nu, gam, s_domain, discrete_num, initial_phi, force_term, "ETD_mrGSAV_MS2_b")
initial_vorticity = np.pad((solver_init.stream2velocity(initial_phi))[0], ((0, 1), (0, 1)))

solver_init.Omega0 = initial_vorticity[:-1, :-1]
solver_init.solve_fix_step((0, 1) , 0.0025)
initial_vorticity = np.pad(solver_init.Omega[-1], ((0, 1), (0, 1)))

In [ ]:
tau_ref = delta*2**-8
solver_ref = vs_mrSAV_solver(nu, gam,s_domain, discrete_num, initial_vorticity, force_term, "ETDRK4")
solver_ref.solve_fix_step(t_period, tau_ref)

In [ ]:
ref_solution_omega = solver_ref.Omega[-1]
ref_solution_psi = solver_ref.vorticity2stream(ref_solution_omega)
ref_u, ref_v = solver_ref.stream2velocity(ref_solution_psi)

reference_data = {
    "omega": ref_solution_omega,
    "psi": ref_solution_psi,
    "u": ref_u,
    "v": ref_v
}

In [ ]:
# 固定步长收敛阶测试
exponents_fixed = [1.0,2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3, 4, 5, 6, 7, 8]
results_fixed = run_convergence_experiment(
    nu, gam, exponents_fixed, initial_vorticity, force_term, reference_data,
    perturb_ratio=0.0, rng_seed=2026,
)

In [ ]:
print("固定步长: vorticity 误差收敛阶")
print_convergence_rates(results_fixed["ETD-mr-SAV-MS2o"], "ETD-mr-SAV-MS2o", component="omega")
print()
print_convergence_rates(results_fixed["mr-SAV-BDF2"], "mr-SAV-BDF2", component="omega")
print()
print_convergence_rates(results_fixed["ETD-MS2"], "ETD-MS2", component="omega")

In [ ]:
df_fixed = make_convergence_display_table(results_fixed, exp_range=(1.0, 3.0))
display(df_fixed.style
    .format({
        "tau":       "{:.2e}",
        "e_u":       "{:.2e}",
        "ord_u":     "{:.2f}",
        "e_omega":   "{:.2e}",
        "ord_omega": "{:.2f}",
        "e_r":       "{:.2e}",
        "ord_r":     "{:.2f}",
    }, na_rep="—")
    .set_caption("Fixed-step convergence errors and orders (exponents 1–3, T=1)  |  blow-up shown as 1e+15")
    .set_table_styles([{"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px")]}])
)


In [ ]:
plot_convergence_results(
    results_fixed,
    save_path='fig/kol_conv_compare_fixed.pdf',
    # title='Fixed-step convergence test',
    title = ""
)


In [ ]:
# 扰动步长收敛阶测试
perturb_ratio = 0.15
perturbed_blowup_value = 1e6
exponents_perturbed = [2, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3, 4, 5, 6, 7, 8]
results_perturbed = run_convergence_experiment(
    nu, gam, exponents_perturbed, initial_vorticity, force_term, reference_data,
    perturb_ratio=perturb_ratio, rng_seed=2026, blowup_value=perturbed_blowup_value,
)

In [ ]:
print(results_perturbed)

In [ ]:
print(f"扰动步长: perturb_ratio = {perturb_ratio}, blowup_value = {perturbed_blowup_value:.1e}")
print_convergence_rates(results_perturbed["ETD-mr-SAV-MS2o"], "ETD-mr-SAV-MS2o", component="omega")
print()
print_convergence_rates(results_perturbed["mr-SAV-BDF2"], "mr-SAV-BDF2", component="omega")
print()
print_convergence_rates(results_perturbed["ETD-MS2"], "ETD-MS2", component="omega")

plot_convergence_results(
    results_perturbed,
    save_path='fig/kol_conv_compare_perturbed.pdf',
    # title=f'Perturbed-step convergence test ($\\epsilon_\\tau={perturb_ratio}$)',
    title="",
    x_key='N',
)


In [ ]:
perturbed_table_latex = make_convergence_latex_table(
    results_perturbed,
    caption=r"Perturbed-step convergence errors and orders at $T=1$.",
    label="tab:convergence_perturbed",
    blowup_value=perturbed_blowup_value,
)
print(perturbed_table_latex)


\begin{exm}\label{exm:kol_conv_test}[Accuracy test]
In this example, $\Omega = (0,2\pi)^2$, $\nu = 1/50$, and the final time is $T = 1$. The initial condition is generated from an isotropic Fourier perturbation of the stream function
\begin{equation}\label{eqn:iso_pertbation}
\psi_{\varepsilon}(x,y) = \varepsilon \sum_{\substack{\bm{k}=(k_1,k_2)\in\mathbb{Z}^2 \\ 0<|\bm{k}|\leq 10}} \frac{1}{|\bm{k}|^3} \bigl[\cos(k_1 x)+\sin(k_1 x)\bigr]\bigl[\cos(k_2 y)+\sin(k_2 y)\bigr],
\end{equation}
with $\varepsilon = 2.5$. The corresponding vorticity field is first evolved to a developed state and then taken as the initial condition $\omega_0$, while the auxiliary variable is initialized by $r_0 = 0$. The initial Reynolds number, estimated from the $L^2$ norm of the induced velocity field, is approximately $1198$, so that the computation is carried out in a moderately turbulent regime. The forcing term in the vorticity equation is given by
\begin{equation*}
f(x,y) = \cos(x).
\end{equation*}
For spatial discretization, $256$ Fourier modes are used. To assess temporal accuracy, we take as reference solution the result computed by the ETDRK4 scheme \cite{kassam2005fourth} with a uniform time step $\tau = 0.01\times 2^{-8}\approx 3.9\times 10^{-5}$.
\end{exm}

We set the mean-reverting parameter $\gamma = 1000$ and $\tilde{\gamma} = 0.1$, and solve the problem by the ETD-mr-SAV-MS2o, mr-SAV-BDF2, and ETD-MS2 schemes. We first consider the fixed-step case. More precisely, we take $\tau = 0.01\times 2^{-k}$ with $k=2,3,\dots,8$, together with several intermediate values between $k=2$ and $k=3$ in order to better resolve the pre-asymptotic regime, and compute at the final time $T=1$ the errors of the velocity, vorticity, and auxiliary variable. The corresponding results are shown in Figure~\ref{fig:convergency_fixed}. One observes that the ETD-mr-SAV-MS2o scheme remains stable over the entire tested range, whereas the ETD-MS2 scheme, which does not incorporate the mean-reverting SAV stabilization, blows up at the coarse step size $\tau = 2.5\times 10^{-3}$. The mr-SAV-BDF2 scheme is unconditionally stable, but its errors are generally larger than those of ETD-mr-SAV-MS2o. For the $L^2$ errors of the velocity and vorticity, all three schemes recover the expected second-order temporal convergence once the step size is sufficiently small. For the auxiliary variable error $|r|$, ETD-mr-SAV-MS2o exhibits first-order convergence, while mr-SAV-BDF2 shows second-order convergence, which is consistent with the distinct treatments of the auxiliary variable in the two formulations.

\begin{figure}[htbp]
\centering
\includegraphics[width=\linewidth]{figure/kol_conv_compare_fixed.pdf}
\caption{The $L^2$ errors of velocity (left) and vorticity (middle), together with the absolute error of the auxiliary variable (right), computed by ETD-mr-SAV-MS2o, ETD-MS2, and mr-SAV-BDF2 at $T=1$ under fixed time steps. The horizontal axis is the time-step size $\tau$, taken from the family $\tau = 0.01\times 2^{-k}$.}
\label{fig:convergency_fixed}
\end{figure}

Next we examine the variable-step setting. Following the same family of average step sizes, we perturb each uniform partition by a relative amplitude of $15\%$ to generate a nonuniform sequence $\{\tau_n\}$ while preserving the final time $T=1$. In this case, the errors are plotted against the total number of time steps $N$, and the results are reported in Figure~\ref{fig:convergency_perturbed}. The numerical results indicate that the ETD-mr-SAV-MS2o and mr-SAV-BDF2 schemes preserve their expected convergence behavior under perturbed time stepping, while ETD-MS2 remains more sensitive on coarse grids and may enter a blow-up regime. Overall, these experiments demonstrate that the proposed mean-reverting SAV treatment not only improves robustness at large step sizes, but also retains the designed convergence order in both fixed-step and variable-step computations.

\begin{figure}[htbp]
\centering
\includegraphics[width=\linewidth]{figure/kol_conv_compare_perturbed.pdf}
\caption{The $L^2$ errors of velocity (left) and vorticity (middle), together with the absolute error of the auxiliary variable (right), computed by ETD-mr-SAV-MS2o, ETD-MS2, and mr-SAV-BDF2 at $T=1$ under a $15\%$ perturbed variable-step sequence. The horizontal axis is the total number of time steps $N$.}
\label{fig:convergency_perturbed}
\end{figure}